# Improving and Evaluating Prompts

---

## Two Distinct Concepts

| | Prompt Engineering | Prompt Evaluation |
|---|---|---|
| **Focus** | How to write better prompts | How to measure if prompts work |
| **Approach** | Techniques and best practices | Automated testing and scoring |
| **Examples** | Multishot prompting, XML tags | Test against expected answers, compare versions, review errors |

---

## Three Paths After Writing a Prompt
```
Draft a Prompt
       |
  _____|______________
  |         |        |
  v         v        v
Option 1  Option 2  Option 3
```

### Option 1 — Test once, ship it
- Test the prompt once and decide it's good enough
- **Risk:** High chance of breaking in production with unexpected inputs

### Option 2 — Test a few times, patch corner cases
- Run it a handful of times and tweak for edge cases you notice
- **Risk:** Users will provide inputs far more varied than what you tested

### Option 3 — Evaluation pipeline (recommended)
- Run the prompt through a scoring pipeline, measure performance, then iterate
- **Trade-off:** More work and cost upfront
- **Benefit:** Much greater confidence in reliability before shipping

---

## Why Engineers Fall Into the First Two Traps

- Options 1 and 2 feel sufficient during development — but production is different
- Real users interact in ways you never anticipated
- Edge cases that seem unlikely become common at scale
- A prompt that works in 10 tests may fail in 1 out of 50 real interactions

---

## The Evaluation-First Approach (Option 3)

Running a prompt through an evaluation pipeline lets you:

- Identify weaknesses **before** they become production issues
- Compare different prompt versions with **objective metrics**
- Iterate with confidence — improvements are **measurable**
- Build applications that are genuinely robust, not just "seems to work"

---

## Key Takeaway

> Writing the prompt is only half the work.  
> Evaluation is what separates a prototype from a production-ready AI feature.

# Prompt Eval Workflow

---

## The 5-Step Process
```
Draft a Prompt
      |
      v
Create an Eval Dataset
      |
      v
Feed Through Claude
      |
      v
Feed Through a Grader
      |
      v
Change Prompt and Repeat
```

---

## Step 1 — Draft a Prompt

Write an initial baseline prompt. This is what you will test and iterate on.
```python
prompt = f"""
Please answer the user's question:

{question}
"""
```

---

## Step 2 — Create an Eval Dataset

A list of sample inputs that represent real-world usage of your prompt.

| Question |
|---|
| What's 2+2? |
| How do I make oatmeal? |
| How far away is the Moon? |

- Can be assembled **by hand** or **generated by Claude**
- In production: tens, hundreds, or thousands of records
- Questions get interpolated into your `{question}` placeholder

---

## Step 3 — Feed Through Claude

Merge each question into the prompt template and send to Claude:
```
Prompt sent:  "Please answer the user's question: What's 2+2?"
Response:     "2 + 2 = 4"

Prompt sent:  "Please answer the user's question: How do I make oatmeal?"
Response:     "Add hot water to dry oatmeal."

Prompt sent:  "Please answer the user's question: How far away is the Moon?"
Response:     "The Moon is about 384,400 KM away."
```

---

## Step 4 — Feed Through a Grader

The grader receives the original question + Claude's answer and assigns a score (1–10).

| Question | Answer | Score |
|---|---|---|
| What's 2+2? | 2 + 2 = 4 | 10 |
| How do I make oatmeal? | Add hot water to dry oatmeal. | 4 |
| How far away is the Moon? | The Moon is about 384,400 KM away. | 9 |

**Average score: (10 + 4 + 9) ÷ 3 = 7.66**

The grader can be another Claude call, a rules-based check, or a human reviewer.

---

## Step 5 — Change Prompt and Repeat

Use the score as a baseline. Modify the prompt and run the full pipeline again.
```python
# Prompt v1 — scored 7.66
prompt = f"""
Please answer the user's question:

{question}
"""

# Prompt v2 — scored 8.7
prompt = f"""
Please answer the user's question:

{question}

Answer the question with ample detail
"""
```

A higher average score confirms the change is an **objective improvement**.

---

## Prompt Scoring — Why It Matters

| Without eval | With eval |
|---|---|
| Gut feeling — "seems better" | Numeric score — objectively better |
| Hard to compare versions | Side-by-side score comparison |
| Guesswork drives changes | Data drives changes |

- Pick the prompt version with the best average score
- Continue iterating until the score meets your quality bar
- Catch regressions early — if a change lowers the score, revert it

---

## Key Takeaway

> The eval workflow turns prompt improvement from guesswork into a **repeatable,
> measurable engineering process** — the same way unit tests give you confidence in code.

# Building a Prompt Eval — Generating the Dataset

---

## Goal

Build an evaluation system for a prompt that helps users write AWS-specific output in
one of three formats — with **no extra explanations, headers, or footers**:

- Python code
- JSON configuration files
- Regular expressions

---

## Baseline Prompt (v1)
````python
prompt = f"""
Please provide a solution to the following task:
{task}
"""
````

This is the prompt we will test and iterate on.

---

## What is an Eval Dataset?

An array of JSON objects, each containing a `task` that gets interpolated into the prompt.
````json
[
  { "task": "Description of task 1" },
  { "task": "Description of task 2" },
  { "task": "Description of task 3" }
]
````

Can be written by hand **or generated automatically using Claude**.  
For generation, use a **faster/cheaper model like Haiku** — no need for full Claude here.

---

## Helper Functions (Recap)
````python
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    response = client.messages.create(**params)
    return response.content[0].text
````

---

## Dataset Generation Function

Uses prefilling + stop sequences to extract clean JSON (no markdown wrapping):
````python
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to
evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks.
Generate an array of JSON objects, each representing a task that requires Python, JSON,
or a Regex to complete.

Example output:
```json
[
  { "task": "Description of task" },
  ...additional
]
```

* Focus on tasks solvable by a single Python function, JSON object, or regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")      # prefill — steer to JSON output
    text = chat(messages, stop_sequences=["```"])   # stop before closing fence

    return json.loads(text)
````

---

## Run and Save the Dataset
````python
import json

# Generate
dataset = generate_dataset()
print(dataset)

# Save to file for reuse across eval runs
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)
````

The saved `dataset.json` will look like:
````json
[
  { "task": "Write a Python function to list all S3 buckets in an account" },
  { "task": "Write a JSON IAM policy that allows read-only access to DynamoDB" },
  { "task": "Write a regex to match a valid AWS ARN" }
]
````

---

## Key Takeaways

- The eval dataset is the foundation — garbage in, garbage out
- Use **Claude Haiku** for dataset generation to save cost
- Use **prefilling + stop sequences** to get clean, parseable JSON
- **Save the dataset to a file** so it stays consistent across eval iterations
- Once saved, every prompt version runs against the exact same inputs

In [7]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [8]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [9]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [10]:
dataset = generate_dataset()

with open("../Artifacts/prompt_eval_dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

# Prompt Eval — Building the Core Pipeline

---

## Overview

The pipeline connects three responsibilities in sequence:
```
dataset.json
     |
     v
run_eval(dataset)          ← loops over all test cases
     |
     v
run_test_case(test_case)   ← runs one case + grades it
     |
     v
run_prompt(test_case)      ← merges prompt + input, calls Claude
```

---

## Function 1 — `run_prompt`

Merges the test case input into the prompt template and returns Claude's output.
```python
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output
```

> This is v1 — no formatting instructions yet. Claude will return verbose output.
> That's intentional: we want to measure the baseline before improving.

---

## Function 2 — `run_test_case`

Calls `run_prompt`, then grades the result. Returns a structured result object.
```python
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO — replace with real grading logic
    score = 10

    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }
```

> Score is hardcoded to 10 for now. Grading logic comes in the next step.

---

## Function 3 — `run_eval`

Loops over the entire dataset, runs each test case, and collects all results.
```python
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results
```

---

## Running the Pipeline
```python
import json

with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)
print(json.dumps(results, indent=2))
```

> First run may take ~30 seconds depending on dataset size and model.

---

## Output Structure

Each result in the returned list contains:
```json
{
  "output": "<Claude's full response>",
  "test_case": { "task": "..." },
  "score": 10
}
```

| Field | Content |
|---|---|
| `output` | Claude's raw response to the merged prompt |
| `test_case` | The original input object from the dataset |
| `score` | Evaluation score (hardcoded for now) |

---

## Current State vs. What's Missing

| Done | Not yet |
|---|---|
| Dataset loading | Real grading logic |
| Prompt + input merging | Formatting instructions in prompt |
| Claude response collection | Score aggregation / averaging |
| Structured result output | Performance optimisation |

---

## Key Takeaway

> This three-function structure — `run_prompt` → `run_test_case` → `run_eval` —
> is the skeleton of virtually every prompt eval pipeline.
> The complexity comes later: better prompts, real graders, and optimisation.
> The hard part is already built.

# Graders — Evaluating Prompt Output Quality

---

## What is a Grader?

A grader takes Claude's output and returns a **measurable score** (typically 1–10).
It replaces the hardcoded `score = 10` placeholder in the eval pipeline.

---

## Three Types of Graders

| Type | How it works | Best for |
|---|---|---|
| **Code** | Programmatic checks in Python | Output length, word presence, syntax validation, readability |
| **Model** | Another Claude/AI call evaluates the output | Response quality, instruction following, completeness, helpfulness, safety |
| **Human** | A person reviews and scores manually | General quality, comprehensiveness, depth, conciseness, relevance |

---

## Evaluation Criteria (for this use case)

Before writing any grader, define what "good output" means:

| Criterion | Description | Best grader type |
|---|---|---|
| **Format** | Returns only Python, JSON, or Regex — no explanation | Code grader |
| **Valid Syntax** | Produced code has valid syntax | Code grader |
| **Task Following** | Response directly and accurately addresses the task | Model grader |

---

## Implementing a Model Grader

Ask the grader model for **strengths, weaknesses, and reasoning** alongside the score.
Without context, models default to middling scores (~6). The reasoning forces deliberate evaluation.
```python
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert code reviewer. Evaluate this AI-generated solution.

Task: {test_case["task"]}
Solution: {output}

Provide your evaluation as a structured JSON object with:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your assessment
- "score": A number between 1-10
"""
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")       # prefill for clean JSON

    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)
```

---

## Integrating the Grader into `run_test_case`

Replace the hardcoded score with a real grader call:
```python
def run_test_case(test_case):
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }
```

---

## Calculating the Average Score in `run_eval`
```python
from statistics import mean

def run_eval(dataset):
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results
```

The average score is your **objective benchmark** — run the eval again after each
prompt change to see if the score goes up or down.

---

## Complete Result Object (per test case)
```json
{
  "output": "<Claude's response>",
  "test_case": { "task": "..." },
  "score": 8,
  "reasoning": "The solution correctly addresses the task and uses valid syntax..."
}
```

---

## Key Takeaways

- Always ask the model grader for **reasoning** — it produces more reliable scores
- Code graders are fast and deterministic; use them for format and syntax checks
- Model graders are flexible but can vary slightly between runs — that's acceptable
- The average score across the dataset is your **single comparable metric** across prompt versions

In [24]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [25]:
# Passes a test case into Claude
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output


# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }
    
# This function coordinates the entire evaluation process:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [26]:
with open("../Artifacts/prompt_eval_dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)
results

Average score: 6.666666666666667


[{'output': '# AWS Region Extractor from S3 URL\n\nHere\'s a Python function that extracts the AWS region from an S3 bucket URL:\n\n```python\nimport re\nfrom urllib.parse import urlparse\n\ndef extract_region_from_s3_url(url):\n    """\n    Extracts the AWS region from an S3 bucket URL.\n    \n    Supports formats:\n    - s3://bucket-name.region.amazonaws.com\n    - https://bucket-name.s3.region.amazonaws.com\n    - https://bucket-name.s3.amazonaws.com (returns \'us-east-1\')\n    - s3://bucket-name\n    \n    Args:\n        url (str): The S3 bucket URL\n        \n    Returns:\n        str: The AWS region name, or None if region cannot be determined\n    """\n    if not url:\n        return None\n    \n    # Remove protocol if present\n    url = url.replace(\'https://\', \'\').replace(\'http://\', \'\').replace(\'s3://\', \'\')\n    \n    # Pattern 1: bucket-name.region.amazonaws.com\n    match = re.search(r\'\\.([a-z0-9\\-]+)\\.amazonaws\\.com\', url)\n    if match:\n        region =

# Code Graders — Syntax Validation

---

## What Code Graders Add

The model grader handles **task following and quality**.  
The code grader handles **format and syntax** — things a model grader is unreliable at.

| Criterion | Grader |
|---|---|
| Format (only Python/JSON/Regex, no explanation) | Code grader |
| Valid syntax | Code grader |
| Task following / accuracy | Model grader |

---

## Syntax Validation Functions

Each function attempts to parse the output as its target format.  
Returns `10` if valid, `0` if not.
```python
import json, ast, re

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0
```

---

## Dataset Format Requirements

Test cases must include a `"format"` field so the grader knows which validator to call:
```json
{
  "task": "Create a Python function to validate an AWS IAM username",
  "format": "python"
}
```

Update your `generate_dataset()` prompt to include `"format"` in the example output structure
so Claude generates it automatically.

---

## `grade_syntax` Dispatcher

Route to the correct validator based on the test case format:
```python
def grade_syntax(output, test_case):
    fmt = test_case.get("format", "").lower()

    if fmt == "json":
        return validate_json(output)
    elif fmt == "python":
        return validate_python(output)
    elif fmt == "regex":
        return validate_regex(output)
    else:
        return 0    # unknown format — fail safe
```

---

## Improving the Prompt for Cleaner Output

Add explicit format instructions to `run_prompt`:
```python
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments, commentary, or explanation
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")    # prefill — encourages raw code output
    output = chat(messages, stop_sequences=["```"])
    return output.strip()
```

---

## Combining Model Score + Syntax Score

Average the two scores for a single composite metric:
```python
def run_test_case(test_case):
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "model_score": model_score,
        "syntax_score": syntax_score,
        "reasoning": model_grade["reasoning"]
    }
```

> Adjust weighting based on your priorities — e.g. `0.4 * model_score + 0.6 * syntax_score`
> if syntactic correctness matters more for your use case.

---

## Implementation Checklist

- Add `validate_json`, `validate_python`, `validate_regex` functions
- Add `"format"` field to dataset test cases
- Update `run_prompt` with explicit format instructions + prefill
- Add `grade_syntax()` dispatcher
- Merge syntax score with model score in `run_test_case`
- Run eval → get baseline average score
- Iterate on prompt → re-run eval → compare scores

---

## Key Takeaway

> The composite score (model grader + code grader) gives you a robust, two-dimensional
> signal: is the output **correct in content** AND **valid in syntax**?
> Use the average score as your benchmark — improve the prompt, re-run, compare.

In [27]:
# Function to generate a new dataset
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [28]:
dataset = generate_dataset()
with open("../Artifacts/prompt_eval_code_check_dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [29]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

# Passes a test case into Claude
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output



In [30]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


In [31]:
# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [32]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [33]:
with open("../Artifacts/prompt_eval_code_check_dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)
print(results)
print(json.dumps(results, indent=2))

Average score: 8.0
[{'output': "\nimport re\n\ndef extract_s3_bucket(uri):\n    match = re.match(r's3://([^/]+)', uri)\n    return match.group(1) if match else None\n", 'test_case': {'task': "Parse an AWS S3 bucket name from an S3 URI in the format 's3://bucket-name/key/path' and extract only the bucket name", 'format': 'regex'}, 'score': 8.5, 'reasoning': 'The solution correctly solves the core task with an appropriate regex pattern that extracts bucket names from properly formatted S3 URIs. However, it lacks robustness for production use. Adding basic input validation, bucket name validation against AWS naming constraints, and documentation would significantly improve reliability. The current implementation works well for valid inputs but provides no feedback for invalid or malformed URIs.'}, {'output': '\n{\n  "MyIAMRole": {\n    "Type": "AWS::IAM::Role",\n    "Properties": {\n      "AssumeRolePolicyDocument": {\n        "Version": "2012-10-17",\n        "Statement": [\n          {\